# Risk model (delivery plan, step 2.3): how well it ranks, and how well it can

The model predicts an unplanned admission in the next 30 days (D37). It was trained once on seed 7 with the
simulator **without a scenario**, so it never saw the hidden patterns or the demo's seed 42
(`uv run python -m engine.risk`). Here it is scored on seed 42, all three scenarios, baseline arm, one
snapshot a week of every enrolled patient out of hospital.

`hazard` is the simulator's own daily hazard on the snapshot day. It is not a feature; it shows the ceiling:
how well anything could rank patients when the outcome is drawn from that hazard.

Re-run: Kernel → Restart and run all (about 2 minutes).

In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

from engine import pathway as pathway_model, risk

hf = pathway_model.load('hf_lombardia_v1')
model = risk.RiskModel(hf)
names = risk.feature_names(hf)
data = {n: risk.dataset(42, n, hf) for n in (1, 2, 3)}
for d in data.values():
    d['p'] = model.booster.predict(xgb.DMatrix(d[names].to_numpy(np.float32), feature_names=names))
print(model.booster.attributes())

{'features': '["age", "distance_km", "lives_alone", "days_since_discharge", "days_since_last_dispense:C03CA01", "on_telemonitoring", "days_since_last_reading", "weight_gain_kg_3d", "overdue_steps", "days_overdue"]', 'horizon_days': '30', 'pathway_id': 'hf_lombardia_v1', 'train_seed': '7', 'version': '1'}


## AUC on the held-out seed, against the simulator's ceiling

In [2]:
pd.DataFrame({f'scenario {n}': {
    'snapshots': len(d), 'admitted within 30 days': round(d.label.mean(), 4),
    'AUC, model': round(roc_auc_score(d.label, d.p), 3),
    'AUC, simulator hazard (ceiling)': round(roc_auc_score(d.label, d.hazard), 3)} for n, d in data.items()})

,scenario 1,scenario 2,scenario 3
snapshots,476053.0000,476021.0000,476068.0000
admitted within 30 days,0.0264,0.0267,0.0265
"AUC, model",0.6190,0.6210,0.6180
"AUC, simulator hazard (ceiling)",0.6170,0.6180,0.6180


The model ranks as well as the simulator's own hazard: a 30-day admission in the simulator is mostly
chance (a low daily hazard drawn every day), so no model can do much better on these data. Real
readmission models report AUCs of about 0.6–0.75.

## Calibration and the high-risk threshold (D37: 0.10)

In [3]:
d = pd.concat(data.values())
d['decile'] = pd.qcut(d.p, 10, labels=False, duplicates='drop')
calib = d.groupby('decile').agg(predicted=('p', 'mean'), observed=('label', 'mean'), snapshots=('p', 'size')).round(4)
rows = []
for t in (0.05, 0.10, 0.15):
    hi = d.p >= t
    rows.append({'threshold': t, 'share of patients': round(hi.mean(), 4), 'admitted': round(d.label[hi].mean(), 4),
                 'lift': round(d.label[hi].mean() / d.label.mean(), 1), 'admissions caught': round(d.label[hi].sum() / d.label.sum(), 4)})
display(calib)
pd.DataFrame(rows).set_index('threshold')

,predicted,observed,snapshots
decile,,,
0,0.0110,0.0158,142817
1,0.0151,0.0170,142842
2,0.0176,0.0188,142785
3,0.0197,0.0200,142863
4,0.0218,0.0223,142767
5,0.0240,0.0239,142811
6,0.0260,0.0238,142818
7,0.0288,0.0287,142810
8,0.0334,0.0330,142820


,share of patients,admitted,lift,admissions caught
threshold,,,,
0.05,0.0428,0.0964,3.6,0.1555
0.10,0.0120,0.1921,7.2,0.0865
0.15,0.0055,0.2699,10.2,0.0558


## What the model uses (gain)

In [4]:
gain = pd.Series(model.booster.get_score(importance_type='gain'))
(gain / gain.sum()).sort_values(ascending=False).round(3).to_frame('share of gain')

,share of gain
weight_gain_kg_3d,0.313
days_since_discharge,0.172
lives_alone,0.132
days_since_last_reading,0.067
age,0.066
distance_km,0.061
days_overdue,0.050
overdue_steps,0.049
days_since_last_dispense:C03CA01,0.047
on_telemonitoring,0.042


## Maria (`p_00001`)

She is 81, lives alone 44.8 km from her hospital and her supply runs out after discharge. Her score stays
near the high-risk line of 0.10 in the weeks after discharge. Her weight rises only from day 43, after the
weekly snapshots below; the pathway rule `weight_gain_kg_3d >= 2` (Monitoring, step 2.8) catches that day.

In [5]:
data[2].loc[data[2].patient_id == 'p_00001', ['day', 'p', 'label', 'days_since_discharge',
                                              'days_since_last_dispense:C03CA01', 'weight_gain_kg_3d']].head(8).round(3)

,day,p,label,days_since_discharge,days_since_last_dispense:C03CA01,weight_gain_kg_3d
0,0,0.229,False,0.0,30.0,NaN
9884,7,0.146,False,7.0,37.0,NaN
19787,14,0.156,False,14.0,44.0,-0.1
29711,21,0.105,True,21.0,51.0,0.3
39598,28,0.107,True,28.0,58.0,-1.3
49487,35,0.094,True,35.0,65.0,0.9
59384,42,0.084,True,42.0,72.0,0.7
79204,56,0.105,False,1.0,0.0,NaN
